# BALANCE Algorithm

## Learning Objectives

1. **Formulate** the Adwords problem as online bipartite matching with budgets
2. **Derive** the BALANCE scoring function $\psi(x) = 1 - e^{x-1}$
3. **State** the competitive ratio $1 - 1/e \approx 0.632$ and its optimality
4. **Compare** BALANCE to greedy (bid only) on adversarial and random inputs
5. **Implement** BALANCE with budget tracking and the $\psi$ weight function


## Problem Statement

### Adwords Problem

Advertisers $\{A_1, \ldots, A_n\}$: each has a total budget $B_i$ and bids $b_{ij}$ for keyword $j$. Queries arrive online; each query $q_t$ triggers a keyword; we must assign $q_t$ to some advertiser $A_i$ who bids on that keyword and has remaining budget $\geq b_{ij}$.

**Goal:** Maximise total revenue $\sum_{(t, i) \text{ matched}} b_{ij}$.

**Simple case:** All bids are equal (= 1) and all budgets are equal. This reduces to online bipartite matching. The greedy algorithm achieves competitive ratio $\frac{1}{2}$.

### BALANCE Algorithm

Instead of assigning to the highest bidder, assign to the advertiser with the **highest adjusted score**:
$$\text{score}(A_i, q_t) = b_{ij} \cdot \psi(x_i)$$
where $x_i = 1 - \frac{\text{remaining budget}_i}{B_i}$ is the fraction of budget $i$ already spent.

The weight function $\psi(x) = 1 - e^{x-1}$:
- $\psi(0) = 1 - e^{-1} \approx 0.632$ (fresh budget)
- $\psi(1) = 0$ (budget exhausted)

This discourages spending the same advertiser's entire budget early.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <defs><marker id="arr" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#999"/></marker></defs>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">BALANCE Algorithm — Budget-Aware Online Matching</text>

  <!-- Setup -->
  <rect x="20" y="36" width="780" height="58" rx="4" fill="#e8f0fb" stroke="#4e79a7"/>
  <text x="410" y="58" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Adwords Setup</text>
  <text x="410" y="76" text-anchor="middle" fill="#333" font-size="11">Advertisers {A₁..Aₙ}: each has total budget Bᵢ and bids bᵢⱼ per keyword j</text>
  <text x="410" y="92" text-anchor="middle" fill="#555" font-size="10">Queries arrive online; assign to at most one advertiser; cannot exceed budget; maximize total revenue</text>

  <!-- Greedy comparison -->
  <rect x="20" y="110" width="360" height="120" rx="4" fill="#fff4e0" stroke="#f28e2b"/>
  <text x="200" y="132" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">Greedy (highest bid wins)</text>
  <text x="200" y="152" text-anchor="middle" fill="#333" font-size="11">Assign to advertiser with max bᵢⱼ</text>
  <text x="200" y="170" text-anchor="middle" fill="#555" font-size="10">Problem: may exhaust rich advertiser's</text>
  <text x="200" y="184" text-anchor="middle" fill="#555" font-size="10">budget early, leaving later queries unmatched</text>
  <text x="200" y="202" text-anchor="middle" fill="#e15759" font-size="11" font-weight="bold">Ratio = 1/2 (worst case)</text>
  <text x="200" y="220" text-anchor="middle" fill="#555" font-size="10">(same as unweighted greedy)</text>

  <!-- BALANCE -->
  <rect x="400" y="110" width="400" height="120" rx="4" fill="#e8f8e8" stroke="#59a14f"/>
  <text x="600" y="132" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">BALANCE: assign by unspent fraction</text>
  <text x="600" y="152" text-anchor="middle" fill="#333" font-size="11">Score(Aᵢ, query j) = bᵢⱼ · ψ(xᵢ)</text>
  <text x="600" y="170" text-anchor="middle" fill="#333" font-size="11">where xᵢ = fraction of budget spent</text>
  <text x="600" y="188" text-anchor="middle" fill="#333" font-size="11">ψ(x) = 1 − e^(x−1)</text>
  <text x="600" y="210" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">Ratio = 1 − 1/e ≈ 0.632</text>
  <text x="600" y="226" text-anchor="middle" fill="#555" font-size="10">(optimal for online bipartite matching)</text>

  <!-- ψ curve -->
  <rect x="20" y="248" width="300" height="104" rx="4" fill="#f5f5f5" stroke="#ccc"/>
  <text x="170" y="268" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">ψ(x) = 1 − e^(x−1)</text>
  <!-- axes -->
  <line x1="40" y1="340" x2="310" y2="340" stroke="#666" stroke-width="1"/>
  <line x1="40" y1="280" x2="40" y2="342" stroke="#666" stroke-width="1"/>
  <text x="310" y="352" text-anchor="middle" fill="#555" font-size="10">x (fraction spent)</text>
  <text x="30" y="284" text-anchor="middle" fill="#555" font-size="10">ψ</text>
  <!-- curve points (x from 0 to 1, mapped to pixel 40-300, y 280-340) -->
  <!-- ψ(0)=1-1/e≈0.632; ψ(1)=0 -->
  <polyline points="40,320 66,316 93,310 120,302 146,293 173,281 200,270 226,257 253,242 280,225 300,212" fill="none" stroke="#59a14f" stroke-width="2"/>
  <!-- correction: ψ(x)=1-e^(x-1), at x=0: 1-e^(-1)≈0.632, at x=1: 0 -->
  <!-- y pixel: 340 - 60*ψ(x), x pixel: 40 + 260*x -->
  <polyline points="40,302 66,299 93,295 120,290 146,284 173,275 200,265 226,252 253,235 280,213 300,198" fill="none" stroke="#59a14f" stroke-width="2.5"/>
  <text x="50" y="306" fill="#555" font-size="9">ψ(0)≈0.63</text>
  <text x="278" y="354" fill="#555" font-size="9">1</text>
  <text x="40" y="354" fill="#555" font-size="9">0</text>

  <!-- Derivation box -->
  <rect x="340" y="248" width="460" height="104" rx="4" fill="#e8f0fb" stroke="#4e79a7"/>
  <text x="570" y="268" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Why 1 − 1/e?</text>
  <text x="570" y="288" text-anchor="middle" fill="#333" font-size="11">Optimal tradeoff: weight unspent budget to discourage</text>
  <text x="570" y="306" text-anchor="middle" fill="#333" font-size="11">spending early. ψ(x)=1−e^(x−1) is the unique function</text>
  <text x="570" y="322" text-anchor="middle" fill="#333" font-size="11">giving competitive ratio exactly 1−1/e for fractional budgets.</text>
  <text x="570" y="340" text-anchor="middle" fill="#555" font-size="10">For integer/non-fractional case, ratio remains 1−1/e (asymptotically).</text>
  <text x="570" y="354" text-anchor="middle" fill="#555" font-size="10">No online algorithm can exceed 1−1/e (KVV 1990 lower bound).</text>
</svg>
'''
display(SVG(svg))


## Derivation

### Competitive Ratio Proof (sketch)

Let OPT be the optimal offline revenue. The BALANCE algorithm is designed so that for every dollar of OPT not matched by BALANCE, at least $1 - (1 - 1/e)$ dollars are captured. Formally:

**Theorem (Mehta et al. 2007):** BALANCE achieves competitive ratio $1 - 1/e$ for the fractional Adwords problem.

**Why $\psi(x) = 1 - e^{x-1}$?**

Choose $\psi$ to satisfy the differential equation that equalises marginal utility across all advertisers at optimum. Solving:
$$\psi'(x) = -e^{x-1} \implies \psi(x) = 1 - e^{x-1}$$

**Why $1 - 1/e$ is optimal:** No online algorithm (deterministic or randomised) can achieve competitive ratio $> 1 - 1/e$ against an adaptive adversary (KVV 1990 lower bound applies to this generalisation).

### Integer vs Fractional

For integer budgets (bids are non-negligible fractions of budget), BALANCE still achieves $1 - 1/e$ asymptotically as bids become small relative to budgets. This is the practical regime (many small clicks vs large budget).


## Algorithm Steps

1. **Maintain** remaining budget $r_i$ for each advertiser; set $\psi(x) = 1 - e^{x-1}$
2. For each arriving query $q_t$ (keyword $j$):
   - Compute eligible set: advertisers with $b_{ij} > 0$ and $r_i \geq b_{ij}$
   - Score: $\text{score}(A_i) = b_{ij} \cdot \psi(1 - r_i / B_i)$
   - Assign to $A^* = \arg\max_i \text{score}(A_i)$; update $r_{A^*} -= b_{A^*,j}$
3. Output matching and total revenue


In [ ]:
import math


def balance(queries, advertisers):
    """
    BALANCE Algorithm for online ad assignment.

    Inputs
    ------
    queries     : list of dicts {'id': str, 'bids': {adv_id: bid_amount}}
                  Queries arrive in order; each has bids from eligible advertisers.
    advertisers : dict {adv_id: {'budget': float}}

    Returns
    -------
    matching    : list of (query_id, adv_id, bid) — matched triples
    total_rev   : float — total revenue
    """
    # Track remaining budget and total budget per advertiser
    remaining = {a: info['budget'] for a, info in advertisers.items()}
    total_bgt  = {a: info['budget'] for a, info in advertisers.items()}

    def psi(x):
        """Weight function: fraction of budget remaining discounted by e^(spend-1)."""
        return 1.0 - math.exp(x - 1.0)

    matching = []
    total_rev = 0.0

    for q in queries:
        best_score = -1.0
        best_adv   = None
        best_bid   = 0.0

        for adv_id, bid in q['bids'].items():
            rem = remaining.get(adv_id, 0.0)
            if rem < bid:
                continue  # cannot afford
            bgt = total_bgt[adv_id]
            x_spent = 1.0 - rem / bgt  # fraction spent so far
            score = bid * psi(x_spent)
            if score > best_score:
                best_score = score
                best_adv   = adv_id
                best_bid   = bid

        if best_adv is not None:
            remaining[best_adv] -= best_bid
            matching.append((q['id'], best_adv, best_bid))
            total_rev += best_bid

    return matching, total_rev


def greedy_highest_bid(queries, advertisers):
    """Baseline: always assign to advertiser with highest bid."""
    remaining = {a: info['budget'] for a, info in advertisers.items()}
    matching = []
    total_rev = 0.0
    for q in queries:
        best_adv, best_bid = None, 0.0
        for adv_id, bid in q['bids'].items():
            if remaining.get(adv_id, 0) >= bid and bid > best_bid:
                best_bid, best_adv = bid, adv_id
        if best_adv:
            remaining[best_adv] -= best_bid
            matching.append((q['id'], best_adv, best_bid))
            total_rev += best_bid
    return matching, total_rev


# ── Demo ──────────────────────────────────────────────────────────────────────
import random; rng = random.Random(42)

advs = {f"A{i}": {"budget": rng.uniform(5, 20)} for i in range(1, 6)}
queries = []
for t in range(40):
    eligible = rng.sample(list(advs.keys()), rng.randint(1, 3))
    bids = {a: rng.uniform(0.5, 3.0) for a in eligible}
    queries.append({"id": f"q{t}", "bids": bids})

m_bal, rev_bal = balance(queries, advs)
m_grd, rev_grd = greedy_highest_bid(queries, advs)

print(f"BALANCE   — matched: {len(m_bal):2d}  revenue: ${rev_bal:.2f}")
print(f"Greedy    — matched: {len(m_grd):2d}  revenue: ${rev_grd:.2f}")
print(f"BALANCE improvement: {(rev_bal/rev_grd - 1)*100:.1f}%")

# Worst-case: two advertisers, equal budget, one dominant bidder
advs2 = {"A1": {"budget": 2.0}, "A2": {"budget": 2.0}}
queries2 = [
    {"id": "q1", "bids": {"A1": 1.0}},             # q1 bids only A1
    {"id": "q2", "bids": {"A1": 1.0}},             # q2 bids only A1
    {"id": "q3", "bids": {"A1": 1.0, "A2": 1.0}},  # q3 bids both
    {"id": "q4", "bids": {"A2": 1.0}},             # q4 bids only A2
]
_, r2 = balance(queries2, advs2)
_, r3 = greedy_highest_bid(queries2, advs2)
print(f"\nSmall example — BALANCE: ${r2:.1f}, Greedy: ${r3:.1f}")
